In [0]:
%sql
CREATE or replace TABLE de_learning_workspace.gold.fact_order_items
(
    order_item_id BIGINT,
    order_id BIGINT,

    customer_sk BIGINT,
    product_sk BIGINT,

    quantity INT,
    unit_price DECIMAL(10,2),
    discount DECIMAL(10,2),
    line_amount DECIMAL(10,2),

    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO de_learning_workspace.gold.fact_order_items
(
    order_item_id,
    order_id,
    customer_sk,
    product_sk,
    quantity,
    unit_price,
    line_amount,
    created_timestamp,
    updated_timestamp
)

SELECT
    oi.order_item_id,
    oi.order_id,

    c.customer_sk,

    p.product_sk,

    oi.quantity,
    oi.unit_price,

    oi.quantity * oi.unit_price,

    current_timestamp(),
    current_timestamp()

FROM de_learning_workspace.silver.order_items oi


JOIN de_learning_workspace.silver.orders o
ON oi.order_id = o.order_id


JOIN de_learning_workspace.gold.dim_customer c
ON o.customer_id = c.customer_id
AND c.is_current = true


JOIN de_learning_workspace.gold.dim_product p
ON oi.product_id = p.product_id
AND p.is_current = true


WHERE NOT EXISTS
(
    SELECT 1
    FROM de_learning_workspace.gold.fact_order_items f
    WHERE f.order_item_id = oi.order_item_id
);